# 05 - XGBoost (+ Random Forest baseline)

Task 3.4: Optuna-tuned XGBoost on the Listing 2.3 feature matrix (Listing 3.4), gain/SHAP feature importance, and an equivalently-tuned Random Forest sanity baseline.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from src import config

In [ ]:
from src.models import tree_models

train = pd.read_parquet(config.PROCESSED_DATA_DIR / "train.parquet")
val = pd.read_parquet(config.PROCESSED_DATA_DIR / "val.parquet")
test = pd.read_parquet(config.PROCESSED_DATA_DIR / "test.parquet")
train_val = pd.concat([train, val])

In [ ]:
xgb_params, study = tree_models.tune_xgboost(train, val, n_trials=30)
xgb_params

In [ ]:
final_model = tree_models.train_xgboost(xgb_params, train_val)
fi = tree_models.feature_importance(final_model, tree_models.get_feature_columns(train_val))
fi.head(10).plot.barh(figsize=(7, 5), title="XGBoost gain-based feature importance")
plt.gca().invert_yaxis()
plt.show()

In [ ]:
shap_values = tree_models.shap_summary(final_model, train_val[tree_models.get_feature_columns(train_val)])
shap_values.abs().mean().sort_values(ascending=False).head(10) if shap_values is not None else "shap unavailable"

## Random Forest baseline (sec. 3.4)

In [ ]:
rf_params, rf_study = tree_models.tune_random_forest(train, val, n_trials=20)
rf_params